In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

BASE_DIR = Path("C:/Users/VSB-AIDSPC20/Downloads/demand forecating")
DATA_PATH = BASE_DIR / "data" / "processed" / "clean_sales.csv"

df = pd.read_csv(DATA_PATH, parse_dates=["date"])
df = df.sort_values(["store", "item", "date"]).reset_index(drop=True)

df.head()


,date,store,item,sales
0,2013-01-01,1,1,13
1,2013-01-02,1,1,11
2,2013-01-03,1,1,14
3,2013-01-04,1,1,13
4,2013-01-05,1,1,10


In [2]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["week"] = df["date"].dt.isocalendar().week.astype(int)
df["day"] = df["date"].dt.day
df["day_of_week"] = df["date"].dt.dayofweek
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)


In [3]:
LAGS = [1, 7, 14, 28]

for lag in LAGS:
    df[f"lag_{lag}"] = (
        df.groupby(["store", "item"])["sales"]
          .shift(lag)
    )


In [4]:
WINDOWS = [7, 14, 28]

for window in WINDOWS:
    df[f"rolling_mean_{window}"] = (
        df.groupby(["store", "item"])["sales"]
          .shift(1)
          .rolling(window)
          .mean()
    )

    df[f"rolling_std_{window}"] = (
        df.groupby(["store", "item"])["sales"]
          .shift(1)
          .rolling(window)
          .std()
    )


In [5]:
df_features = df.dropna().reset_index(drop=True)
df_features.shape


(899000, 20)

In [6]:
SPLIT_DATE = "2017-01-01"

train = df_features[df_features["date"] < SPLIT_DATE]
valid = df_features[df_features["date"] >= SPLIT_DATE]

X_train = train.drop(columns=["sales", "date"])
y_train = train["sales"]

X_valid = valid.drop(columns=["sales", "date"])
y_valid = valid["sales"]


In [7]:
OUTPUT_DIR = BASE_DIR / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train.to_csv(OUTPUT_DIR / "train_features.csv", index=False)
valid.to_csv(OUTPUT_DIR / "valid_features.csv", index=False)

print("Feature-engineered datasets saved.")


Feature-engineered datasets saved.
